# Microsoft Harrier Embedding for conversion and quantized script

In [ ]:
!apt update -y
!apt install -y build-essential cmake git-lfs libcur14-openssl-dev

!pip install -U huggingface_hub hf_xet transformers sentencepiece protobuf accelerate safetensors

In [ ]:
from huggingface_hub import login, HfApi

login()


api = HfApi()
print(f"Logged in as: {api.whoami()['name']}")

In [ ]:
from huggingface_hub import snapshot_download
model_id = "microsoft/harrier-oss-v1-0.6b"
local_dir = "/content/harrier-oss-v1-0.6b"

snapshot_download(repo_id=model_id,
                  cache_dir=local_dir,
                  local_dir=local_dir,
                  local_dir_use_symlinks=False)

print("Download to:", local_dir)

In [ ]:
!ls -lh /content/harrier-oss-v1-0.6b

In [ ]:
%cd /content
!rm -rf llama.cpp
!git clone https://github.com/ggml-org/llama.cpp.git

%cd /content/llama.cpp

!pip install -r requirements.txt
!cmake -B build
!cmake --build build --config Release -j 2

In [ ]:
%cd /content

!python /content/llama.cpp/convert_hf_to_gguf.py \
 /content/harrier-oss-v1-0.6b \
 --outfile /content/harrier-oss-v1-0.6b-F16.gguf \
 --outtype f16

In [ ]:
!ls -lh /content/*.gguf

In [ ]:
! /content/llama.cpp/build/bin/llama-quantize \
   /content/harrier-oss-v1-0.6b-F16.gguf \
   /content/harrier-oss-v1-0.6b-Q8_0.gguf \
   Q8_0

In [ ]:
!ls -lh /content/*.gguf

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

# Get current username to ensure correct namespace
username = api.whoami()["name"]
repo_id = f"{username}/harrier-oss-v1-0.6b-GGUF"

print(f"Target repository: {repo_id}")

# Create the repo
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)

# Upload the GGUF files
files_to_upload = [
    "harrier-oss-v1-0.6b-F16.gguf",
    "harrier-oss-v1-0.6b-Q8_0.gguf"
]

for file_name in files_to_upload:
    print(f"Uploading {file_name}...")
    api.upload_file(
        path_or_fileobj=f"/content/{file_name}",
        path_in_repo=file_name,
        repo_id=repo_id
    )

print(f"All files successfully uploaded to: https://huggingface.co/{repo_id}")

In [ ]:
! /content/llama.cpp/build/bin/llama-quantize \
   /content/harrier-oss-v1-0.6b-F16.gguf \
   /content/harrier-oss-v1-0.6b-Q4_K_M.gguf \
   Q4_K_M

In [ ]:
!ls -lh /content/*.gguf

In [ ]:
api.upload_file(
    path_or_fileobj=f"/content/harrier-oss-v1-0.6b-Q4_K_M.gguf",
    path_in_repo="harrier-oss-v1-0.6b-Q4_K_M.gguf",
    repo_id=repo_id,
    repo_type="model"
)